# 0 · Data Check — HH data volume

Goal: inventory the **HH** batch before vs. after the rebuild, to compare sizes.

- **Legacy artifacts (before)**: `optimized_HH_chunks/`, `optimized_HH_embeddings/`
- **New-pipeline input**: `converted_raw_txts/` (already includes HH + Qs)
- **New output (after chunking)**: chunk count + volume of the HH subset in the rebuilt table — run after `1_build_chunks`

> This repo must be used as a **Databricks Repo**: the repo root is auto-added to `sys.path`, so `from src... import` works.

In [0]:
from src import config as C
from src.stats import fs_stats, jsonl_chunk_stats, count_doc_folders, table_stats, bytes_human

## 1) Legacy HH artifacts · optimized_HH_chunks (file count + size)

In [0]:
# HH legacy chunks: doc folders, chunk records, raw jsonl volume
hh_docs   = count_doc_folders(spark, C.HH_OPT_CHUNKS)
hh_chunks = jsonl_chunk_stats(spark, C.HH_OPT_CHUNKS + "*/chunks.jsonl")
hh_vol    = fs_stats(spark, C.HH_OPT_CHUNKS, glob="*.jsonl", recursive=True)

print("optimized_HH_chunks")
print("  doc folders   :", hh_docs)
print("  chunk records :", hh_chunks["n_chunks"])
print("  source docs   :", hh_chunks["n_docs"])
print("  jsonl files   :", hh_vol["n_files"])
print("  volume        :", bytes_human(hh_vol["total_bytes"]))

## 2) Legacy HH artifacts · optimized_HH_embeddings (file count + size)

In [0]:
hh_emb = fs_stats(spark, C.HH_OPT_EMB, glob="*", recursive=True)
print("optimized_HH_embeddings")
print("  files  :", hh_emb["n_files"])
print("  volume :", bytes_human(hh_emb["total_bytes"]))

## 3) New-pipeline input · converted_raw_txts (ALL, includes HH + Qs)

In [0]:
src_all = fs_stats(spark, C.SRC_TXT_DIR, glob="*.txt", recursive=True)
print("converted_raw_txts (ALL)")
print("  txt files :", src_all["n_files"])
print("  volume    :", bytes_human(src_all["total_bytes"]))

## 4) New output (after chunking) — run after `1_build_chunks`

Count chunks + volume of the **HH subset** in the rebuilt table, to compare against the
legacy `optimized_HH` numbers above. `HH_TAG` is the substring identifying HH files in
`source_file`/path — placeholder for now, adjust once confirmed.

In [0]:
from pyspark.sql import functions as F

HH_TAG = "HH"  # TODO: confirm how HH files are identified in source_file / path

# Run AFTER 1_build_chunks has written C.TBL_CHUNKS
chunks = spark.table(C.TBL_CHUNKS)
hh_new = chunks.filter(F.col("source_file").contains(HH_TAG))
print("rebuilt HH chunks :", hh_new.count())

tv = table_stats(spark, C.TBL_CHUNKS)
print(C.TBL_CHUNKS, "-> rows:", tv["n_rows"], "| size:", bytes_human(tv["size_bytes"]))